# Training a fruit-sorting neuron

This notebook goes with the Detailed lesson **Neural Networks: From One Neuron to Deep Learning**, section 5.

A single neuron sorts fruit using sweetness and roundness. We score three fruits, apply the perceptron learning rule, and train until every fruit is sorted correctly. Run each cell in order with **Shift + Enter**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# (sweetness, roundness), label: 1 = apple, 0 = lemon
fruits = {"apple": ((9, 8), 1), "lemon": ((2, 4), 0), "mystery fruit": ((7, 3), 0)}

## 1. Scoring with the starting weights

In [ ]:
w = np.array([1.2, 0.8])
b = -10.0

def score(x, w, b):
    return float(np.dot(w, x) + b)

def predict(x, w, b):
    return 1 if score(x, w, b) >= 0 else 0

for name, (x, label) in fruits.items():
    z = score(x, w, b)
    print(f"{name:13s} z = {z:5.1f} -> {'apple' if z >= 0 else 'lemon'} (correct: {'apple' if label else 'lemon'})")

## 2. One learning step

The mystery fruit belongs with the lemons, but the neuron says apple. Apply the rule with learning rate 0.1: each weight changes by 0.1 × (label − prediction) × input.

In [ ]:
rate = 0.1
x, label = fruits["mystery fruit"]
error = label - predict(x, w, b)
w = w + rate * error * np.array(x)
b = b + rate * error
print("error:", error, " new weights:", w, " new bias:", round(b, 2))

for name, (x, label) in fruits.items():
    print(f"{name:13s} z = {score(x, w, b):5.1f}")

Fixing the mystery fruit has pushed the apple below zero. Training continues over all the fruits until nothing is wrong.

## 3. The full training loop

In [ ]:
def train(data, w, b, rate=0.1, max_epochs=100):
    for epoch in range(1, max_epochs + 1):
        errors = 0
        for name, (x, label) in data.items():
            error = label - predict(x, w, b)
            if error:
                errors += 1
                w = w + rate * error * np.array(x)
                b = b + rate * error
                print(f"epoch {epoch}: {name} wrong, w = {np.round(w, 2)}, b = {b:.2f}")
        if errors == 0:
            print(f"epoch {epoch}: every fruit correct, training stops")
            return w, b
    print("did not settle within", max_epochs, "epochs")
    return w, b

w, b = train(fruits, np.array([1.2, 0.8]), -10.0)

## 4. Drawing the boundary

The neuron says apple on one side of the line w₁ × sweetness + w₂ × roundness + b = 0.

In [ ]:
xs = np.linspace(0, 10, 50)
plt.figure(figsize=(5, 5))
for name, ((s, r), label) in fruits.items():
    plt.scatter(s, r, s=120, color="tab:red" if label else "gold", edgecolor="black")
    plt.annotate(name, (s, r), textcoords="offset points", xytext=(6, 6))
if w[1] != 0:
    plt.plot(xs, -(w[0] * xs + b) / w[1], "k--", label="boundary")
plt.xlim(0, 10); plt.ylim(0, 10)
plt.xlabel("sweetness"); plt.ylabel("roundness"); plt.legend(); plt.title("Learned boundary")
plt.show()

## 5. A problem one neuron cannot solve

XOR is true when exactly one input is true. No single straight line separates its cases, so the perceptron never settles.

In [ ]:
xor = {"0,0": ((0, 0), 0), "0,1": ((0, 1), 1), "1,0": ((1, 0), 1), "1,1": ((1, 1), 0)}
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    w_xor, b_xor = train(xor, np.zeros(2), 0.0, max_epochs=50)
wrong = sum(predict(x, w_xor, b_xor) != label for x, label in xor.values())
print("After 50 epochs,", wrong, "of 4 XOR cases are still wrong.")

## 6. Try it yourself

- Change the learning rate to 1.0 or 0.01. How many epochs does training take?
- Add a fourth fruit, such as a sweet, round orange that you label as a lemon. Can one neuron still sort them all?